# 연도별 시드 문맥 감성 수준 분석

## 분석 목적

이 노트북의 목적은 v_2_1_seed_term_context_sentiment.ipynb에서 생성한 seed-only 문맥 감성 변수를 사용해, 긍정/부정 문맥의 수준(level)이 연도별 ESG 등급과 어떻게 관련되는지 확인하는 것이다.

이 분석은 전년 대비 변화량이 아니라 기업-연도별 수준값을 본다. 사업보고서의 fiscal_year는 다음 해 ESG 평가연도와 연결해 esg_year = fiscal_year + 1 기준으로 분석한다.

## 입력

- final/v_2_1_seed_term_context_sentiment_analysis.csv

## 출력

- final/v_2_4_yearly_seed_context_sentiment_spearman.csv
- final/v_2_4_yearly_seed_context_sentiment_ols.csv
- final/v_2_4_yearly_seed_context_sentiment_group_summary.csv
- final/v_2_4_yearly_seed_context_sentiment_kruskal.csv


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, kruskal
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 160)

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)
FINAL_DIR = ROOT / "final"
INPUT_PATH = first_existing([
    FINAL_DIR / "v_2_1_seed_term_context_sentiment_analysis.csv",
    LOCAL_ROOT / "final" / "v_2_1_seed_term_context_sentiment_analysis.csv",
    LOCAL_ROOT / "v_2_1_seed_term_context_sentiment_analysis.csv",
])

print("ROOT:", ROOT)
print("INPUT_PATH:", INPUT_PATH, "|", "OK" if INPUT_PATH.exists() else "MISSING")


ROOT: /content/drive/MyDrive/UD_26
INPUT_PATH: /content/drive/MyDrive/UD_26/final/v_2_1_seed_term_context_sentiment_analysis.csv | OK


In [2]:
analysis_df = pd.read_csv(INPUT_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
analysis_df["stock_code"] = analysis_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year", "esg_grade_num"]:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

if "esg_year" not in analysis_df.columns:
    analysis_df["esg_year"] = analysis_df["fiscal_year"] + 1

if not (analysis_df["esg_year"].dropna().astype(int) == analysis_df.loc[analysis_df["esg_year"].notna(), "fiscal_year"].astype(int) + 1).all():
    raise ValueError("Expected ESG year to be fiscal_year + 1.")

print("analysis_df:", analysis_df.shape)
print("missing esg_grade_num:", analysis_df["esg_grade_num"].isna().sum())
print("fiscal_year -> esg_year pairs")
display(analysis_df.groupby(["fiscal_year", "esg_year"]).size().rename("n").reset_index())
display(analysis_df.head())


analysis_df: (378, 38)
missing esg_grade_num: 0
fiscal_year -> esg_year pairs


,fiscal_year,esg_year,n
0,2022,2023,126
1,2023,2024,126
2,2024,2025,126


,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,seed_sentence_count,positive_seed_sentence_count,negative_seed_sentence_count,neutral_seed_sentence_count,mean_seed_sentence_sentiment,unique_seed_terms_in_sentences,seed_term_context_count,seed_term_occurrence_count,positive_seed_term_context_count,negative_seed_term_context_count,neutral_seed_term_context_count,mean_seed_term_context_sentiment,unique_seed_terms_matched,positive_seed_sentence_share,negative_seed_sentence_share,neutral_seed_sentence_share,positive_seed_term_context_share,negative_seed_term_context_share,seed_sentence_per_1000_words,seed_term_context_per_1000_words,seed_term_occurrence_per_1000_words,industry,esg_grade,e_grade,s_grade,g_grade,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
0,000020,동화약품,2022,2023,20230315001100,7808,37953,3,35,2,2,31,0.018317,14,68,118,2,3,63,-0.000832,14,0.057143,0.057143,0.885714,0.029412,0.044118,4.482582,8.709016,15.112705,NaN,C,C,B,C,1,1,2,1
1,000020,동화약품,2023,2024,20240319000652,8065,38467,3,35,2,1,32,0.036592,13,67,114,2,2,63,0.008703,13,0.057143,0.028571,0.914286,0.029851,0.029851,4.339740,8.307502,14.135152,NaN,C,B,B,C,1,2,2,1
2,000020,동화약품,2024,2025,20250318000739,8097,39259,3,37,2,0,35,0.053468,14,68,115,2,0,66,0.029093,14,0.054054,0.000000,0.945946,0.029412,0.000000,4.569594,8.398172,14.202791,NaN,C,B,C,C,1,2,1,1
3,000040,KR모터스,2022,2023,20230322001182,4201,20136,3,11,1,0,10,0.090888,10,18,38,1,0,17,0.055543,10,0.090909,0.000000,0.909091,0.055556,0.000000,2.618424,4.284694,9.045465,NaN,D,D,D,D,0,0,0,0
4,000040,KR모터스,2023,2024,20240321002062,4888,22402,3,11,0,1,10,-0.047889,11,16,38,0,1,15,-0.032923,11,0.000000,0.090909,0.909091,0.000000,0.062500,2.250409,3.273322,7.774141,NaN,D,D,D,D,0,0,0,0


In [3]:
# Derived level variables. Shares capture sentiment composition, while counts capture sentiment volume.
analysis_df = analysis_df.copy()

required_base_cols = [
    "positive_seed_sentence_count", "negative_seed_sentence_count", "seed_sentence_count",
    "positive_seed_term_context_count", "negative_seed_term_context_count", "seed_term_context_count",
    "total_word_count", "esg_grade_num", "fiscal_year", "esg_year",
]
missing = [col for col in required_base_cols if col not in analysis_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

analysis_df["net_seed_sentence_count"] = analysis_df["positive_seed_sentence_count"] - analysis_df["negative_seed_sentence_count"]
analysis_df["net_seed_sentence_share"] = analysis_df["positive_seed_sentence_share"] - analysis_df["negative_seed_sentence_share"]
analysis_df["net_seed_term_context_count"] = analysis_df["positive_seed_term_context_count"] - analysis_df["negative_seed_term_context_count"]
analysis_df["net_seed_term_context_share"] = analysis_df["positive_seed_term_context_share"] - analysis_df["negative_seed_term_context_share"]

analysis_df["positive_seed_sentence_per_1000_words"] = 1000 * analysis_df["positive_seed_sentence_count"] / analysis_df["total_word_count"].replace(0, np.nan)
analysis_df["negative_seed_sentence_per_1000_words"] = 1000 * analysis_df["negative_seed_sentence_count"] / analysis_df["total_word_count"].replace(0, np.nan)
analysis_df["positive_seed_term_context_per_1000_words"] = 1000 * analysis_df["positive_seed_term_context_count"] / analysis_df["total_word_count"].replace(0, np.nan)
analysis_df["negative_seed_term_context_per_1000_words"] = 1000 * analysis_df["negative_seed_term_context_count"] / analysis_df["total_word_count"].replace(0, np.nan)

for col in [
    "positive_seed_sentence_share", "negative_seed_sentence_share",
    "positive_seed_term_context_share", "negative_seed_term_context_share",
]:
    analysis_df[f"c_{col}"] = analysis_df[col] - analysis_df[col].mean(skipna=True)

analysis_df["sentence_share_interaction"] = analysis_df["c_positive_seed_sentence_share"] * analysis_df["c_negative_seed_sentence_share"]
analysis_df["term_context_share_interaction"] = analysis_df["c_positive_seed_term_context_share"] * analysis_df["c_negative_seed_term_context_share"]

summary_cols = [
    "positive_seed_sentence_count", "negative_seed_sentence_count", "net_seed_sentence_count",
    "positive_seed_sentence_share", "negative_seed_sentence_share", "net_seed_sentence_share",
    "positive_seed_term_context_count", "negative_seed_term_context_count", "net_seed_term_context_count",
    "positive_seed_term_context_share", "negative_seed_term_context_share", "net_seed_term_context_share",
    "seed_sentence_count", "seed_term_context_count", "total_word_count", "esg_grade_num",
]
display(analysis_df[summary_cols].describe().T)


,count,mean,std,min,25%,50%,75%,max
positive_seed_sentence_count,378.0,9.214286,14.414262,0.000000,2.000000,5.000000,10.000000,118.000000
negative_seed_sentence_count,378.0,2.775132,4.139442,0.000000,1.000000,2.000000,3.000000,50.000000
net_seed_sentence_count,378.0,6.439153,11.513699,-7.000000,1.000000,3.000000,8.000000,83.000000
positive_seed_sentence_share,378.0,0.083938,0.065005,0.000000,0.036785,0.067606,0.125910,0.326241
negative_seed_sentence_share,378.0,0.029213,0.023426,0.000000,0.012987,0.026316,0.040816,0.142857
net_seed_sentence_share,378.0,0.054725,0.066650,-0.142857,0.007663,0.039088,0.101271,0.295918
positive_seed_term_context_count,378.0,12.042328,19.904368,0.000000,2.000000,6.000000,12.000000,164.000000
negative_seed_term_context_count,378.0,4.111111,9.982655,0.000000,1.000000,2.000000,4.000000,126.000000
net_seed_term_context_count,378.0,7.931217,14.280834,-9.000000,0.000000,3.000000,9.000000,96.000000
positive_seed_term_context_share,378.0,0.063339,0.053247,0.000000,0.025000,0.048669,0.091355,0.252874


In [4]:
def spearman_for_feature(data, y_col, x_col):
    tmp = data[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 3 or tmp[x_col].nunique() < 2 or tmp[y_col].nunique() < 2:
        return len(tmp), np.nan, np.nan
    rho, p_value = spearmanr(tmp[x_col], tmp[y_col])
    return len(tmp), float(rho), float(p_value)


SPEARMAN_FEATURES = [
    "positive_seed_sentence_count",
    "negative_seed_sentence_count",
    "net_seed_sentence_count",
    "positive_seed_sentence_share",
    "negative_seed_sentence_share",
    "net_seed_sentence_share",
    "mean_seed_sentence_sentiment",
    "positive_seed_sentence_per_1000_words",
    "negative_seed_sentence_per_1000_words",
    "positive_seed_term_context_count",
    "negative_seed_term_context_count",
    "net_seed_term_context_count",
    "positive_seed_term_context_share",
    "negative_seed_term_context_share",
    "net_seed_term_context_share",
    "mean_seed_term_context_sentiment",
    "positive_seed_term_context_per_1000_words",
    "negative_seed_term_context_per_1000_words",
    "seed_sentence_count",
    "seed_term_context_count",
    "seed_term_occurrence_count",
    "unique_seed_terms_matched",
    "total_word_count",
]

spearman_rows = []
for fiscal_year, year_df in analysis_df.groupby("fiscal_year", dropna=True):
    esg_years = sorted(year_df["esg_year"].dropna().unique().tolist())
    esg_year = esg_years[0] if len(esg_years) == 1 else esg_years
    for feature in SPEARMAN_FEATURES:
        if feature not in year_df.columns:
            continue
        n, rho, p_value = spearman_for_feature(year_df, "esg_grade_num", feature)
        spearman_rows.append({
            "fiscal_year": int(fiscal_year),
            "esg_year": esg_year,
            "feature": feature,
            "n": n,
            "spearman_rho": rho,
            "abs_spearman_rho": abs(rho) if pd.notna(rho) else np.nan,
            "p_value": p_value,
        })

spearman_yearly_df = pd.DataFrame(spearman_rows).sort_values(
    ["fiscal_year", "abs_spearman_rho"], ascending=[True, False]
).reset_index(drop=True)

display(spearman_yearly_df)

print("Top features by year")
display(spearman_yearly_df.groupby("fiscal_year").head(10).reset_index(drop=True))


,fiscal_year,esg_year,feature,n,spearman_rho,abs_spearman_rho,p_value
0,2022,2023,seed_term_occurrence_count,126,0.742587,0.742587,2.477617e-23
1,2022,2023,seed_term_context_count,126,0.710669,0.710669,1.153926e-20
2,2022,2023,total_word_count,126,0.677328,0.677328,3.083901e-18
3,2022,2023,seed_sentence_count,126,0.646802,0.646802,2.804399e-16
4,2022,2023,unique_seed_terms_matched,126,0.609395,0.609395,3.669669e-14
...,...,...,...,...,...,...,...
64,2024,2025,positive_seed_term_context_per_1000_words,126,0.226243,0.226243,1.085344e-02
65,2024,2025,negative_seed_term_context_share,126,0.064849,0.064849,4.706396e-01
66,2024,2025,negative_seed_term_context_per_1000_words,126,0.058712,0.058712,5.137362e-01
67,2024,2025,negative_seed_sentence_share,126,0.053681,0.053681,5.505099e-01


Top features by year


,fiscal_year,esg_year,feature,n,spearman_rho,abs_spearman_rho,p_value
0,2022,2023,seed_term_occurrence_count,126,0.742587,0.742587,2.477617e-23
1,2022,2023,seed_term_context_count,126,0.710669,0.710669,1.153926e-20
2,2022,2023,total_word_count,126,0.677328,0.677328,3.083901e-18
3,2022,2023,seed_sentence_count,126,0.646802,0.646802,2.804399e-16
4,2022,2023,unique_seed_terms_matched,126,0.609395,0.609395,3.669669e-14
5,2022,2023,positive_seed_term_context_count,126,0.594501,0.594501,2.148886e-13
6,2022,2023,positive_seed_sentence_count,126,0.577051,0.577051,1.523290e-12
7,2022,2023,net_seed_sentence_count,126,0.511875,0.511875,9.012921e-10
8,2022,2023,net_seed_term_context_count,126,0.510097,0.510097,1.053240e-09
9,2022,2023,positive_seed_sentence_share,126,0.365228,0.365228,2.610595e-05


In [5]:
def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def run_robust_ols(data, y_col, x_cols, min_n=20):
    required_cols = [y_col] + x_cols
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        return None, {"skip_reason": f"missing columns: {missing_cols}", "n": 0}
    reg_df = data[required_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < min_n:
        return None, {"skip_reason": f"too few complete rows after dropna: {len(reg_df)}", "n": int(len(reg_df))}
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return model, {"skip_reason": "", "n": int(model.nobs)}


MODEL_SPECS = {
    "Y1_sentence_pos_neg_counts": [
        "positive_seed_sentence_count", "negative_seed_sentence_count",
    ],
    "Y2_sentence_pos_neg_counts_length": [
        "positive_seed_sentence_count", "negative_seed_sentence_count", "total_word_count",
    ],
    "Y3_sentence_pos_neg_shares_volume": [
        "positive_seed_sentence_share", "negative_seed_sentence_share", "seed_sentence_count", "total_word_count",
    ],
    "Y4_sentence_net_share_volume": [
        "net_seed_sentence_share", "seed_sentence_count", "total_word_count",
    ],
    "Y5_sentence_share_interaction": [
        "positive_seed_sentence_share", "negative_seed_sentence_share", "sentence_share_interaction", "seed_sentence_count", "total_word_count",
    ],
    "Y6_context_pos_neg_counts": [
        "positive_seed_term_context_count", "negative_seed_term_context_count", "unique_seed_terms_matched",
    ],
    "Y7_context_pos_neg_counts_length": [
        "positive_seed_term_context_count", "negative_seed_term_context_count", "unique_seed_terms_matched", "total_word_count",
    ],
    "Y8_context_pos_neg_shares_volume": [
        "positive_seed_term_context_share", "negative_seed_term_context_share", "seed_term_context_count", "total_word_count",
    ],
    "Y9_context_net_share_volume": [
        "net_seed_term_context_share", "seed_term_context_count", "total_word_count",
    ],
    "Y10_context_share_interaction": [
        "positive_seed_term_context_share", "negative_seed_term_context_share", "term_context_share_interaction", "seed_term_context_count", "total_word_count",
    ],
}

ols_rows = []
for fiscal_year, year_df in analysis_df.groupby("fiscal_year", dropna=True):
    esg_years = sorted(year_df["esg_year"].dropna().unique().tolist())
    esg_year = esg_years[0] if len(esg_years) == 1 else esg_years
    for model_name, x_cols in MODEL_SPECS.items():
        model, info = run_robust_ols(year_df, "esg_grade_num", x_cols, min_n=20)
        if model is None:
            ols_rows.append({
                "fiscal_year": int(fiscal_year),
                "esg_year": esg_year,
                "model": model_name,
                "variable": "SKIPPED",
                **info,
            })
            continue
        for variable in model.params.index:
            ols_rows.append({
                "fiscal_year": int(fiscal_year),
                "esg_year": esg_year,
                "model": model_name,
                "variable": variable,
                "coef": float(model.params[variable]),
                "std_err": float(model.bse[variable]),
                "p_value": float(model.pvalues[variable]),
                "r2": float(model.rsquared),
                "n": int(model.nobs),
                "skip_reason": "",
            })

ols_yearly_df = pd.DataFrame(ols_rows)
display(ols_yearly_df)

print("Compact non-constant yearly OLS results")
compact_ols_yearly_df = ols_yearly_df[ols_yearly_df["variable"].ne("const")].sort_values(
    ["fiscal_year", "model", "p_value"], ascending=[True, True, True]
).reset_index(drop=True)
display(compact_ols_yearly_df)


,fiscal_year,esg_year,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,2022,2023,Y1_sentence_pos_neg_counts,const,2.555556,0.142433,5.525654e-72,0.153346,126,
1,2022,2023,Y1_sentence_pos_neg_counts,positive_seed_sentence_count,0.782696,0.322820,1.532702e-02,0.153346,126,
2,2022,2023,Y1_sentence_pos_neg_counts,negative_seed_sentence_count,-0.203342,0.483934,6.743493e-01,0.153346,126,
3,2022,2023,Y2_sentence_pos_neg_counts_length,const,2.555556,0.130572,2.677074e-85,0.274232,126,
4,2022,2023,Y2_sentence_pos_neg_counts_length,positive_seed_sentence_count,0.278605,0.355051,4.326365e-01,0.274232,126,
...,...,...,...,...,...,...,...,...,...,...
133,2024,2025,Y10_context_share_interaction,positive_seed_term_context_share,-0.029516,0.140165,8.332169e-01,0.295545,126,
134,2024,2025,Y10_context_share_interaction,negative_seed_term_context_share,-0.188181,0.180890,2.981978e-01,0.295545,126,
135,2024,2025,Y10_context_share_interaction,term_context_share_interaction,-0.132037,0.293230,6.525047e-01,0.295545,126,
136,2024,2025,Y10_context_share_interaction,seed_term_context_count,0.831739,0.355503,1.930362e-02,0.295545,126,


Compact non-constant yearly OLS results


,fiscal_year,esg_year,model,variable,coef,std_err,p_value,r2,n,skip_reason
0,2022,2023,Y10_context_share_interaction,seed_term_context_count,0.862246,0.239815,0.000324,0.370360,126,
1,2022,2023,Y10_context_share_interaction,negative_seed_term_context_share,-0.267254,0.145657,0.066532,0.370360,126,
2,2022,2023,Y10_context_share_interaction,total_word_count,0.238926,0.173118,0.167545,0.370360,126,
3,2022,2023,Y10_context_share_interaction,term_context_share_interaction,-0.169249,0.233866,0.469250,0.370360,126,
4,2022,2023,Y10_context_share_interaction,positive_seed_term_context_share,0.073474,0.155719,0.637042,0.370360,126,
...,...,...,...,...,...,...,...,...,...,...
103,2024,2025,Y8_context_pos_neg_shares_volume,total_word_count,0.192996,0.183443,0.292764,0.290580,126,
104,2024,2025,Y8_context_pos_neg_shares_volume,positive_seed_term_context_share,-0.015085,0.137182,0.912439,0.290580,126,
105,2024,2025,Y9_context_net_share_volume,seed_term_context_count,0.621101,0.286871,0.030381,0.279433,126,
106,2024,2025,Y9_context_net_share_volume,total_word_count,0.229535,0.182557,0.208634,0.279433,126,


In [6]:
# Year-specific high/low grouping: within each fiscal year, compare high/low positive and negative sentiment levels.

def yearwise_high_low(series):
    median = series.median(skipna=True)
    return np.where(series >= median, "high", "low")

analysis_df["positive_sentence_group"] = analysis_df.groupby("fiscal_year")["positive_seed_sentence_share"].transform(yearwise_high_low)
analysis_df["negative_sentence_group"] = analysis_df.groupby("fiscal_year")["negative_seed_sentence_share"].transform(yearwise_high_low)
analysis_df["pos_neg_sentence_group"] = "pos_" + analysis_df["positive_sentence_group"] + "__neg_" + analysis_df["negative_sentence_group"]

analysis_df["positive_context_group"] = analysis_df.groupby("fiscal_year")["positive_seed_term_context_share"].transform(yearwise_high_low)
analysis_df["negative_context_group"] = analysis_df.groupby("fiscal_year")["negative_seed_term_context_share"].transform(yearwise_high_low)
analysis_df["pos_neg_context_group"] = "pos_" + analysis_df["positive_context_group"] + "__neg_" + analysis_df["negative_context_group"]

GROUP_SPECS = {
    "sentence_share_high_low": {
        "group_col": "pos_neg_sentence_group",
        "positive_col": "positive_seed_sentence_share",
        "negative_col": "negative_seed_sentence_share",
    },
    "term_context_share_high_low": {
        "group_col": "pos_neg_context_group",
        "positive_col": "positive_seed_term_context_share",
        "negative_col": "negative_seed_term_context_share",
    },
}

group_rows = []
kruskal_rows = []
for fiscal_year, year_df in analysis_df.groupby("fiscal_year", dropna=True):
    esg_years = sorted(year_df["esg_year"].dropna().unique().tolist())
    esg_year = esg_years[0] if len(esg_years) == 1 else esg_years
    for analysis_name, spec in GROUP_SPECS.items():
        group_col = spec["group_col"]
        positive_col = spec["positive_col"]
        negative_col = spec["negative_col"]
        summary = (
            year_df.groupby(group_col)
            .agg(
                n=("esg_grade_num", "count"),
                mean_esg_grade=("esg_grade_num", "mean"),
                median_esg_grade=("esg_grade_num", "median"),
                mean_positive_level=(positive_col, "mean"),
                mean_negative_level=(negative_col, "mean"),
            )
            .reset_index()
        )
        summary.insert(0, "analysis", analysis_name)
        summary.insert(0, "esg_year", esg_year)
        summary.insert(0, "fiscal_year", int(fiscal_year))
        group_rows.append(summary)

        groups = [g["esg_grade_num"].dropna().values for _, g in year_df.groupby(group_col) if len(g["esg_grade_num"].dropna()) > 0]
        if len(groups) >= 2:
            stat, p_value = kruskal(*groups)
            kruskal_rows.append({
                "fiscal_year": int(fiscal_year),
                "esg_year": esg_year,
                "analysis": analysis_name,
                "group_col": group_col,
                "statistic": float(stat),
                "p_value": float(p_value),
            })
        else:
            kruskal_rows.append({
                "fiscal_year": int(fiscal_year),
                "esg_year": esg_year,
                "analysis": analysis_name,
                "group_col": group_col,
                "statistic": np.nan,
                "p_value": np.nan,
            })

group_summary_df = pd.concat(group_rows, ignore_index=True).sort_values(
    ["fiscal_year", "analysis", "mean_esg_grade"], ascending=[True, True, False]
)
kruskal_yearly_df = pd.DataFrame(kruskal_rows)

display(group_summary_df)
display(kruskal_yearly_df)


,fiscal_year,esg_year,analysis,pos_neg_sentence_group,n,mean_esg_grade,median_esg_grade,mean_positive_level,mean_negative_level,pos_neg_context_group
1,2022,2023,sentence_share_high_low,pos_high__neg_low,28,3.035714,3.0,0.131383,0.011598,NaN
0,2022,2023,sentence_share_high_low,pos_high__neg_high,35,2.942857,4.0,0.135684,0.046823,NaN
2,2022,2023,sentence_share_high_low,pos_low__neg_high,28,2.392857,3.0,0.040740,0.046126,NaN
3,2022,2023,sentence_share_high_low,pos_low__neg_low,35,1.914286,1.0,0.030724,0.008952,NaN
4,2022,2023,term_context_share_high_low,NaN,33,3.121212,4.0,0.106027,0.041133,pos_high__neg_high
5,2022,2023,term_context_share_high_low,NaN,30,3.100000,3.5,0.096736,0.007855,pos_high__neg_low
6,2022,2023,term_context_share_high_low,NaN,30,2.133333,2.0,0.028758,0.039735,pos_low__neg_high
7,2022,2023,term_context_share_high_low,NaN,33,1.878788,1.0,0.020465,0.005694,pos_low__neg_low
9,2023,2024,sentence_share_high_low,pos_high__neg_low,28,3.500000,4.0,0.133633,0.014603,NaN
8,2023,2024,sentence_share_high_low,pos_high__neg_high,35,3.085714,4.0,0.140270,0.043336,NaN


,fiscal_year,esg_year,analysis,group_col,statistic,p_value
0,2022,2023,sentence_share_high_low,pos_neg_sentence_group,9.563861,0.022662
1,2022,2023,term_context_share_high_low,pos_neg_context_group,14.242409,0.002593
2,2023,2024,sentence_share_high_low,pos_neg_sentence_group,21.263988,0.000093
3,2023,2024,term_context_share_high_low,pos_neg_context_group,23.587078,0.000030
4,2024,2025,sentence_share_high_low,pos_neg_sentence_group,13.929593,0.003003
5,2024,2025,term_context_share_high_low,pos_neg_context_group,11.920402,0.007661


In [7]:
# Focus table: core positive/negative coefficients by year.
CORE_VARIABLES = [
    "positive_seed_sentence_share",
    "negative_seed_sentence_share",
    "positive_seed_sentence_count",
    "negative_seed_sentence_count",
    "positive_seed_term_context_share",
    "negative_seed_term_context_share",
    "positive_seed_term_context_count",
    "negative_seed_term_context_count",
    "net_seed_sentence_share",
    "net_seed_term_context_share",
]

core_ols_yearly_df = compact_ols_yearly_df[compact_ols_yearly_df["variable"].isin(CORE_VARIABLES)].copy()
core_ols_yearly_df["direction"] = np.where(core_ols_yearly_df["coef"] > 0, "+", np.where(core_ols_yearly_df["coef"] < 0, "-", "0"))
core_ols_yearly_df["significant_0_05"] = core_ols_yearly_df["p_value"] < 0.05

display(core_ols_yearly_df.sort_values(["fiscal_year", "model", "variable"]))

core_spearman_yearly_df = spearman_yearly_df[spearman_yearly_df["feature"].isin(CORE_VARIABLES)].copy()
core_spearman_yearly_df["direction"] = np.where(core_spearman_yearly_df["spearman_rho"] > 0, "+", np.where(core_spearman_yearly_df["spearman_rho"] < 0, "-", "0"))
core_spearman_yearly_df["significant_0_05"] = core_spearman_yearly_df["p_value"] < 0.05

display(core_spearman_yearly_df.sort_values(["fiscal_year", "feature"]))


,fiscal_year,esg_year,model,variable,coef,std_err,p_value,r2,n,skip_reason,direction,significant_0_05
1,2022,2023,Y10_context_share_interaction,negative_seed_term_context_share,-0.267254,0.145657,0.066532,0.370360,126,,-,False
4,2022,2023,Y10_context_share_interaction,positive_seed_term_context_share,0.073474,0.155719,0.637042,0.370360,126,,+,False
6,2022,2023,Y1_sentence_pos_neg_counts,negative_seed_sentence_count,-0.203342,0.483934,0.674349,0.153346,126,,-,False
5,2022,2023,Y1_sentence_pos_neg_counts,positive_seed_sentence_count,0.782696,0.322820,0.015327,0.153346,126,,+,True
9,2022,2023,Y2_sentence_pos_neg_counts_length,negative_seed_sentence_count,-0.205856,0.352234,0.558931,0.274232,126,,-,False
8,2022,2023,Y2_sentence_pos_neg_counts_length,positive_seed_sentence_count,0.278605,0.355051,0.432636,0.274232,126,,+,False
12,2022,2023,Y3_sentence_pos_neg_shares_volume,negative_seed_sentence_share,-0.203952,0.121681,0.093714,0.332969,126,,-,False
11,2022,2023,Y3_sentence_pos_neg_shares_volume,positive_seed_sentence_share,0.245823,0.132709,0.063977,0.332969,126,,+,False
14,2022,2023,Y4_sentence_net_share_volume,net_seed_sentence_share,0.284947,0.128081,0.026098,0.329021,126,,+,True
20,2022,2023,Y5_sentence_share_interaction,negative_seed_sentence_share,-0.207771,0.136106,0.126877,0.337595,126,,-,False


,fiscal_year,esg_year,feature,n,spearman_rho,abs_spearman_rho,p_value,direction,significant_0_05
17,2022,2023,negative_seed_sentence_count,126,0.304543,0.304543,5.260012e-04,+,True
19,2022,2023,negative_seed_sentence_share,126,0.052139,0.052139,5.620361e-01,+,False
18,2022,2023,negative_seed_term_context_count,126,0.285397,0.285397,1.198294e-03,+,True
21,2022,2023,negative_seed_term_context_share,126,0.044790,0.044790,6.184762e-01,+,False
12,2022,2023,net_seed_sentence_share,126,0.342921,0.342921,8.466787e-05,+,True
14,2022,2023,net_seed_term_context_share,126,0.316071,0.316071,3.116537e-04,+,True
6,2022,2023,positive_seed_sentence_count,126,0.577051,0.577051,1.523290e-12,+,True
9,2022,2023,positive_seed_sentence_share,126,0.365228,0.365228,2.610595e-05,+,True
5,2022,2023,positive_seed_term_context_count,126,0.594501,0.594501,2.148886e-13,+,True
10,2022,2023,positive_seed_term_context_share,126,0.348372,0.348372,6.402616e-05,+,True


In [8]:
OUTPUT_SPEARMAN_PATH = FINAL_DIR / "v_2_4_yearly_seed_context_sentiment_spearman.csv"
OUTPUT_OLS_PATH = FINAL_DIR / "v_2_4_yearly_seed_context_sentiment_ols.csv"
OUTPUT_CORE_OLS_PATH = FINAL_DIR / "v_2_4_yearly_seed_context_sentiment_core_ols.csv"
OUTPUT_CORE_SPEARMAN_PATH = FINAL_DIR / "v_2_4_yearly_seed_context_sentiment_core_spearman.csv"
OUTPUT_GROUP_SUMMARY_PATH = FINAL_DIR / "v_2_4_yearly_seed_context_sentiment_group_summary.csv"
OUTPUT_KRUSKAL_PATH = FINAL_DIR / "v_2_4_yearly_seed_context_sentiment_kruskal.csv"

spearman_yearly_df.to_csv(OUTPUT_SPEARMAN_PATH, index=False, encoding="utf-8-sig")
ols_yearly_df.to_csv(OUTPUT_OLS_PATH, index=False, encoding="utf-8-sig")
core_ols_yearly_df.to_csv(OUTPUT_CORE_OLS_PATH, index=False, encoding="utf-8-sig")
core_spearman_yearly_df.to_csv(OUTPUT_CORE_SPEARMAN_PATH, index=False, encoding="utf-8-sig")
group_summary_df.to_csv(OUTPUT_GROUP_SUMMARY_PATH, index=False, encoding="utf-8-sig")
kruskal_yearly_df.to_csv(OUTPUT_KRUSKAL_PATH, index=False, encoding="utf-8-sig")

print("saved spearman:", OUTPUT_SPEARMAN_PATH, spearman_yearly_df.shape)
print("saved ols:", OUTPUT_OLS_PATH, ols_yearly_df.shape)
print("saved core ols:", OUTPUT_CORE_OLS_PATH, core_ols_yearly_df.shape)
print("saved core spearman:", OUTPUT_CORE_SPEARMAN_PATH, core_spearman_yearly_df.shape)
print("saved group summary:", OUTPUT_GROUP_SUMMARY_PATH, group_summary_df.shape)
print("saved kruskal:", OUTPUT_KRUSKAL_PATH, kruskal_yearly_df.shape)


saved spearman: /content/drive/MyDrive/UD_26/final/v_2_4_yearly_seed_context_sentiment_spearman.csv (69, 7)
saved ols: /content/drive/MyDrive/UD_26/final/v_2_4_yearly_seed_context_sentiment_ols.csv (138, 10)
saved core ols: /content/drive/MyDrive/UD_26/final/v_2_4_yearly_seed_context_sentiment_core_ols.csv (54, 12)
saved core spearman: /content/drive/MyDrive/UD_26/final/v_2_4_yearly_seed_context_sentiment_core_spearman.csv (30, 9)
saved group summary: /content/drive/MyDrive/UD_26/final/v_2_4_yearly_seed_context_sentiment_group_summary.csv (24, 10)
saved kruskal: /content/drive/MyDrive/UD_26/final/v_2_4_yearly_seed_context_sentiment_kruskal.csv (6, 6)
